# Segmentation performance by lesion size (simulation)

Train two 48-tree pixel classifiers using RGB and local intensity features at 48 × 48 resolution. Compare uniform pixel sampling against equal foreground/background sampling. Each seed uses 400 training and 200 held-out images. Define small lesions using the median training-mask area and report image-level mean Dice in both strata. Each seed generates 600 RGB phantoms with variable disk radii, lesion contrast and pixel noise. The foreground mask is known exactly.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'lesion-size-segmentation-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0', 'Pillow==12.3.0']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom scipy import ndimage as ndi\nfrom sklearn.ensemble import ExtraTreesClassifier\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", str(ROOT / "data")))\nSEEDS = [13, 37, 61]\n\n\ndef save(model, name):\n    path = ROOT / "checkpoints"\n    path.mkdir(exist_ok=True)\n    joblib.dump(model, path / (name + ".joblib"), compress=3)\n\n\ndef examples(seed):\n    rng = np.random.default_rng(seed)\n    u, v = np.mgrid[-1:1:48j, -1:1:48j]\n    imgs = []\n    masks = []\n    for _ in range(600):\n        a, b = rng.uniform(-0.45, 0.45, 2)\n        m = ((u - a) ** 2 + (v - b) ** 2) < rng.uniform(0.08, 0.48) ** 2\n        noise = rng.normal(0.4, 0.17, (48, 48, 3))\n        imgs.append(\n            np.clip(noise + m[:, :, None] * np.array([0.19, 0.10, -0.05]), 0, 1)\n        )\n        masks.append(m)\n    return np.array(imgs), np.array(masks)\n\n\ndef features(im):\n    gray = im.mean(2)\n    return np.dstack(\n        [\n            im,\n            ndi.gaussian_filter(gray, 2),\n            ndi.gaussian_gradient_magnitude(gray, 1),\n            ndi.uniform_filter(gray, 9),\n        ]\n    ).reshape(-1, 6)\n\n\ndef run(seed):\n    x, y = examples(seed)\n    rng = np.random.default_rng(seed)\n    a = np.stack([features(img) for img in x])\n    sizes = y.mean((1, 2))\n    thresh = np.median(sizes[:400])\n    out = {}\n    for method in ["uniform", "balanced_foreground"]:\n        inputs = []\n        labels = []\n        for z, m in zip(a[:400], y[:400]):\n            flat = m.ravel()\n            idx = rng.choice(len(flat), 160)\n            if method == "balanced_foreground" and flat.any() and (~flat).any():\n                idx = np.r_[\n                    rng.choice(np.flatnonzero(flat), 80),\n                    rng.choice(np.flatnonzero(~flat), 80),\n                ]\n            inputs.append(z[idx])\n            labels.append(flat[idx])\n        model = ExtraTreesClassifier(\n            n_estimators=48, min_samples_leaf=5, n_jobs=2, random_state=seed\n        ).fit(np.vstack(inputs), np.concatenate(labels))\n        save(model, f"{method}-{seed}")\n        p = model.predict_proba(a[400:].reshape(-1, 6))[:, 1].reshape(-1, 48, 48)\n        pred = p >= 0.5\n        truth = y[400:]\n        dice = (2 * (pred & truth).sum((1, 2)) + 1) / (\n            pred.sum((1, 2)) + truth.sum((1, 2)) + 1\n        )\n        small = sizes[400:] <= thresh\n        out[method] = {\n            "small_dice": float(dice[small].mean()),\n            "large_dice": float(dice[~small].mean()),\n            "all_dice": float(dice.mean()),\n        }\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())